In [0]:
from pyspark.sql.functions import col, count, min as spark_min, max as spark_max

# 1. Variables de configuration du compte de stockage
storage_account_name = "stairqualitydevnu81f1"
storage_account_key = dbutils.secrets.get(scope="kv-scope", key="stairqualitydevnu81f1-key")

# 2. Chemins des conteneurs Azure
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/openaq/air_quality_cleaned"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/openaq/kpi_stations_by_country"

# 3. Lecture de la couche Silver (Format Delta)
df_silver = (
    spark.read
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
    .format("delta")
    .load(silver_path)
)

# 4. Transformations Gold : Agrégations par pays
df_gold = (
    df_silver
    .groupBy("country_code")
    .agg(
        count("location_id").alias("total_stations"),
        spark_min("first_updated_at").alias("oldest_station_date"),
        spark_max("last_updated_at").alias("latest_activity_date")
    )
    .orderBy(col("total_stations").desc())
)

# 5. Affichage du résultat analytique
display(df_gold)

# 6. Écriture au format Delta Lake dans le conteneur Gold
(
    df_gold.write
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .save(gold_path)
)

print("✅ Couche Gold (KPIs par pays) générée et sauvegardée avec succès en Delta Lake !")